# C6 · Alineamiento, BLAST+ local y MAFFT

**Curso:** Bioinformática y Biología Computacional · Universidad EAFIT  
**Duración sugerida:** 3 horas  
**Modalidad:** explicación breve → práctica guiada → reto → evidencia reproducible

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UniversidadEAFIT/compubiol_course/blob/master/notebooks/06_alineamiento_blast/06_alineamiento_blast_mafft.ipynb)

> **Continuidad del material histórico:** esta versión reemplaza y amplía `20231/blasting/Blasting.ipynb`.

## Pregunta guía

Se necesita identificar miembros plausibles de una familia génica. **¿Cómo distinguir similitud útil, homología inferida y función demostrada?**

### Objetivos

- diferenciar alineamiento global/local y seleccionar `blastn`, `blastp`, `blastx`, `tblastn` o `tblastx`;
- interpretar identidad, cobertura, E-value, longitud y bitscore conjuntamente;
- construir una base local y ejecutar BLAST+;
- definir criterios antes de seleccionar resultados;
- crear y revisar un alineamiento múltiple con MAFFT;
- identificar secuencias truncadas, baja complejidad y falsos positivos.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/UniversidadEAFIT/compubiol_course.git"
COLAB_DIR = Path("/content/compubiol_course")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and importlib.util.find_spec("Bio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

if IN_COLAB and not COLAB_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_DIR)], check=True)
    os.chdir(COLAB_DIR)

start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del curso. Ejecute el notebook desde el repositorio clonado."
    )
os.chdir(ROOT)
os.environ["COURSE_ROOT"] = str(ROOT)
print(f"Raíz del curso: {ROOT}")

## 1. La pregunta determina el programa

| Consulta | Base | Programa |
|---|---|---|
| nucleótido | nucleótido | `blastn` |
| proteína | proteína | `blastp` |
| nucleótido traducido | proteína | `blastx` |
| proteína | nucleótido traducido | `tblastn` |
| nucleótido traducido | nucleótido traducido | `tblastx` |

La homología es una hipótesis histórica binaria; “75 % homólogo” es una formulación incorrecta. Reporte porcentaje de identidad y cobertura, y use el conjunto de evidencia para inferir homología.

## 2. Global y local en un ejemplo pequeño

In [ ]:
from Bio.Align import PairwiseAligner

query = "MKTAYIAKQRQISFVKSHFSRQ"
target = "GGGMKTAYIAKQRQISFVKSHFSRQAAA"

for mode in ["global", "local"]:
    aligner = PairwiseAligner(mode=mode)
    aligner.match_score = 2
    aligner.mismatch_score = -1
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -0.5
    alignment = aligner.align(query, target)[0]
    print(f"\n{mode.upper()} score={alignment.score}")
    print(alignment)

## 3. Dataset sintético y criterios

In [ ]:
from Bio import SeqIO

query_record = next(SeqIO.parse(ROOT / "data/module06/query.faa", "fasta"))
db_records = list(SeqIO.parse(ROOT / "data/module06/protein_db.faa", "fasta"))
print("Consulta:", query_record.id, len(query_record.seq), "aa")
for record in db_records:
    print(f"{record.id:24s} {len(record.seq):3d} aa")

Antes de ejecutar, escriba una política inicial, por ejemplo:

- identidad ≥ 35 %;
- cobertura de consulta ≥ 60 %;
- E-value ≤ 1e-3;
- longitud compatible;
- exclusión o inspección de baja complejidad;
- validación con dominios, contexto y filogenia.

Los umbrales no son universales: dependen de longitud, divergencia, base y propósito.

## 4. Flujo local reproducible

In [ ]:
import shutil, subprocess

tools = {name: shutil.which(name) for name in ["makeblastdb", "blastp", "mafft"]}
print(tools)
if all(tools.values()):
    result = subprocess.run(
        ["bash", "scripts/module06/run_blast.sh"],
        cwd=ROOT, text=True, capture_output=True
    )
    print("exit:", result.returncode)
    print(result.stdout)
    print(result.stderr)
else:
    print("Instale el ambiente completo para ejecutar BLAST+/MAFFT. El script está listo para terminal o HPC.")

In [ ]:
import pandas as pd
from pathlib import Path

blast_table = ROOT / "results/module06/blast_all.tsv"
if blast_table.exists():
    hits = pd.read_csv(blast_table, sep="\t")
    display(hits.sort_values("bitscore", ascending=False))
else:
    print("La tabla aparecerá después de ejecutar scripts/module06/run_blast.sh")

### Checkpoint

- Alta identidad con cobertura baja puede indicar un fragmento o dominio compartido.
- E-value depende del tamaño de la base y la puntuación; no reemplaza identidad/cobertura.
- Bitscore permite comparar calidad de alineamiento dentro del mismo esquema de puntuación.
- Un mejor hit no demuestra función ni ortología.

## 5. Alineamiento múltiple

In [ ]:
from Bio import AlignIO
alignment_path = ROOT / "data/module07/homologs_aligned.faa"
aln = AlignIO.read(alignment_path, "fasta")
print("Secuencias:", len(aln), "Columnas:", aln.get_alignment_length())
print(aln[:, :60])

Revise: secuencias excesivamente cortas, bloques sin homología clara, largas inserciones, redundancia, encabezados y cobertura del dominio relevante. La salida de MAFFT no debe aceptarse automáticamente.

## Reto

Aplique el flujo a una familia NAC vegetal: seleccione una consulta bien anotada, justifique programa y base, defina criterios, conserve falsos positivos y prepare el MSA para C7.

Referencias: NCBI BLAST+ manual; Altschul et al. (1997); documentación y artículo de MAFFT.